# Dependências

In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login


In [1]:
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling


c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [2]:
diretorio = 'G:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\415 - Repositório de Dados\\Repositório Local\\PNAD'

In [3]:
os.chdir(diretorio)  

In [4]:
os.listdir(diretorio)

['2025', '2024', 'pnad_indicadores_serie_2016_2025.xlsx']

In [5]:
df1 = pd.read_excel('pnad_indicadores_serie_2016_2025.xlsx', sheet_name='indicador_01')
df1

,ano,sexo,freq,freq_se,freq_cv,prop,prop_se,prop_cv
0,2016,Homem,4.650154e+06,60894.072647,0.013095,0.438430,0.003871,0.008829
1,2016,Mulher,5.956221e+06,75429.441968,0.012664,0.561570,0.003871,0.006893
2,2017,Homem,4.555177e+06,71551.347587,0.015708,0.434586,0.003788,0.008716
3,2017,Mulher,5.926472e+06,67586.336967,0.011404,0.565414,0.003788,0.006700
4,2018,Homem,4.763627e+06,62369.660775,0.013093,0.439744,0.003912,0.008896
5,2018,Mulher,6.069093e+06,73108.988493,0.012046,0.560256,0.003912,0.006982
6,2019,Homem,4.817930e+06,61934.873475,0.012855,0.438725,0.003897,0.008883
7,2019,Mulher,6.163729e+06,80631.135466,0.013082,0.561275,0.003897,0.006944
8,2020,Homem,4.911042e+06,85462.328913,0.017402,0.438152,0.004740,0.010818
9,2020,Mulher,6.297487e+06,78979.991956,0.012542,0.561848,0.004740,0.008436


In [6]:
df = pd.read_excel('pnad_indicadores_serie_2016_2025.xlsx', sheet_name='indicador_05')
df.columns

Index(['ano', 'sexo', 'cor', 'freq', 'freq_se', 'freq_cv', 'prop', 'prop_se',
       'prop_cv'],
      dtype='object')

In [9]:
df[df['ano'] == 2025][df['sexo'] == 'Homem']['freq'].sum()

C:\Users\ana.sales_republica\AppData\Local\Temp\ipykernel_9764\1841366076.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['ano'] == 2025][df['sexo'] == 'Homem']['freq'].sum()


5162300.480368051

In [10]:
df["freq"] = df["freq"].round().astype("Int64")
df = df.drop(columns=['freq_se', 'freq_cv','prop_se', 'prop_cv'])
df["prop"] = (df["prop"] * 100).round(2)
df = df.rename(columns= {'freq':'quantidade_vinculos', 'sexo':'genero','cor':'cor_raca'})   
df = df[['ano', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df

,ano,cor_raca,genero,quantidade_vinculos,prop
0,2016,Branca,Homem,2271963,48.86
1,2016,Negra,Homem,2311141,49.70
2,2016,Outra,Homem,67002,1.44
3,2016,Branca,Mulher,3147663,52.85
4,2016,Negra,Mulher,2762851,46.39
5,2016,Outra,Mulher,45708,0.77
6,2017,Branca,Homem,2212551,48.58
7,2017,Negra,Homem,2282327,50.11
8,2017,Outra,Homem,60025,1.32
9,2017,Branca,Mulher,3068556,51.78


In [ ]:
df1['ano'] = 2024
df1["sum"] = df1["sum"].round().astype("Int64")
df1 = df1.drop(columns=['cv', 'Unnamed: 6'])
df1 = df1.rename(columns= {'sum':'quantidade_vinculos', 'cor':'cor_raca','sexo':'genero'}) 
df1 = df1[['ano','esfera', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df1


# Upload

In [101]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [104]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('esfera', 'STRING', description= 'Nível da esfera do governo referente da observação'),
          bigquery.SchemaField('cor_raca', 'STRING', description= 'Raça/cor autodeclarado ou não'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('perfil_remuneracao')

table_ref = dataset_ref.table('PNAD_vinculos_genero_cor') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_merged, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=994c9731-4334-4523-8b23-f7882a4e2860>